<a href="https://colab.research.google.com/github/mwafa2/Further-Pretraining/blob/main/MGB-5-FullFT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# ── Install gdown ──────────────────────────────────────────────────────────
!pip install -q gdown

# ── Download processor folder from Drive ──────────────────────────────────
!gdown --folder "https://drive.google.com/drive/folders/13E7EyY97Ta4YRt62DJ5MaQJb7mySkQk8?usp=sharing" -O /content/processor

# ── Verify files downloaded ────────────────────────────────────────────────
import os
print("Processor files:", os.listdir("/content/processor"))

# ── Load dataset and processor ─────────────────────────────────────────────
from datasets import load_dataset
from transformers import Wav2Vec2Processor

dataset = load_dataset("ArabicSpeech/MGB-5")
processor = Wav2Vec2Processor.from_pretrained("/content/processor")

print(dataset)
print(processor)

Retrieving folder contents
Processing file 1wbuK6SZjS14PuLzoul7EMjInzXpa22g6 added_tokens.json
Processing file 1YT5YQhDJx3dEOyiew06hMeYJ2Tsp8CVi processor_config.json
Processing file 1xLdKLV-zzt_CHp6g0QbQFAiiusncmhG5 tokenizer_config.json
Processing file 1QK3rfIvtyYKhiUPLBqFEQjW2PVKpMHJs vocab.json
Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=1wbuK6SZjS14PuLzoul7EMjInzXpa22g6
To: /content/processor/added_tokens.json
100% 30.0/30.0 [00:00<00:00, 165kB/s]
Downloading...
From: https://drive.google.com/uc?id=1YT5YQhDJx3dEOyiew06hMeYJ2Tsp8CVi
To: /content/processor/processor_config.json
100% 299/299 [00:00<00:00, 1.78MB/s]
Downloading...
From: https://drive.google.com/uc?id=1xLdKLV-zzt_CHp6g0QbQFAiiusncmhG5
To: /content/processor/tokenizer_config.json
100% 1.28k/1.28k [00:00<00:00, 8.79MB/s]
Downloading...
From: https://drive.google.com/uc?id=1QK3rfIvtyYKhiUPLBqFEQjW2PVKpMHJs
To: 

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


data/train.parquet:   0%|          | 0.00/4.82G [00:00<?, ?B/s]

data/dev.parquet:   0%|          | 0.00/890M [00:00<?, ?B/s]

data/test.parquet:   0%|          | 0.00/945M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/33338 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/6163 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5752 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'audio', 'text'],
        num_rows: 33338
    })
    validation: Dataset({
        features: ['id', 'audio', 'text'],
        num_rows: 6163
    })
    test: Dataset({
        features: ['id', 'audio', 'text'],
        num_rows: 5752
    })
})
Wav2Vec2Processor:
- feature_extractor: Wav2Vec2FeatureExtractor {
  "do_normalize": true,
  "feature_extractor_type": "Wav2Vec2FeatureExtractor",
  "feature_size": 1,
  "padding_side": "right",
  "padding_value": 0.0,
  "return_attention_mask": true,
  "sampling_rate": 16000
}

- tokenizer: Wav2Vec2CTCTokenizer(name_or_path='/content/processor', vocab_size=36, model_max_length=1000000000000000019884624838656, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<s>', 'eos_token': '</s>', 'unk_token': '[UNK]', 'pad_token': '[PAD]', 'word_delimiter_token': '|'}, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=True, lstrip=True, single_word=False, normalized=F

In [3]:
from datasets import load_dataset

dataset = load_dataset("ArabicSpeech/MGB-5")

print(dataset)
print("\nColumn names:", dataset["train"].column_names)
print("\nFirst sample keys:", dataset["train"][0].keys())
print("\nFirst sample (non-audio):")
for k, v in dataset["train"][0].items():
    if k != "audio":
        print(f"  {k}: {v}")
print("\nAudio info:", {k: v for k, v in dataset["train"][0]["audio"].items() if k != "array"})

DatasetDict({
    train: Dataset({
        features: ['id', 'audio', 'text'],
        num_rows: 33338
    })
    validation: Dataset({
        features: ['id', 'audio', 'text'],
        num_rows: 6163
    })
    test: Dataset({
        features: ['id', 'audio', 'text'],
        num_rows: 5752
    })
})

Column names: ['id', 'audio', 'text']

First sample keys: dict_keys(['id', 'audio', 'text'])

First sample (non-audio):
  id: Comedy_21Spn2w7UOI_annotator_2_101.71_108.696
  text: لا والو هداك السيد جارنا بروفوكاني بغا يحكرني ولكن نهار لاول يموت لمش دخلتو سوق راسو


AttributeError: 'AudioDecoder' object has no attribute 'items'

In [4]:
import numpy as np

# ── Check audio info properly ──────────────────────────────────────────────
sample = dataset["train"][0]
audio = sample["audio"]
print("Audio type:", type(audio))
print("Audio sampling rate:", audio["sampling_rate"])
print("Audio array length:", len(audio["array"]))
print("Duration (sec):", len(audio["array"]) / audio["sampling_rate"])
print("Text:", sample["text"])

# ── Tokenizer sanity check ─────────────────────────────────────────────────
encoded = processor.tokenizer(sample["text"])
print("\nToken IDs:", encoded.input_ids[:30])
print("Decoded:", processor.tokenizer.decode(encoded.input_ids, group_tokens=False))

Audio type: <class 'datasets.features._torchcodec.AudioDecoder'>
Audio sampling rate: 16000
Audio array length: 111776
Duration (sec): 6.986
Text: لا والو هداك السيد جارنا بروفوكاني بغا يحكرني ولكن نهار لاول يموت لمش دخلتو سوق راسو

Token IDs: [29, 6, 2, 33, 6, 29, 33, 2, 32, 14, 6, 28, 2, 6, 29, 18, 35, 14, 2, 11, 6, 16, 31, 6, 2, 7, 16, 33, 26, 33]
Decoded: لا والو هداك السيد جارنا بروفوكاني بغا يحكرني ولكن نهار لاول يموت لمش دخلتو سوق راسو


In [5]:
from dataclasses import dataclass
from typing import Any, Dict, List, Union
import torch

# ── Feature extraction + label preparation ─────────────────────────────────
def prepare_dataset(batch):
    expected_sr = processor.feature_extractor.sampling_rate
    actual_sr   = batch["audio"][0]["sampling_rate"]
    assert actual_sr == expected_sr, \
        f"Expected sampling rate {expected_sr}, but got {actual_sr}"

    audio_arrays = [a["array"] for a in batch["audio"]]

    processed_audio = processor(
        audio_arrays,
        sampling_rate=expected_sr
    )

    batch["input_values"] = processed_audio.input_values
    batch["input_length"] = [len(x) for x in processed_audio.input_values]
    batch["labels"]       = processor.tokenizer(batch["text"]).input_ids  # MGB-5 uses "text" column

    return batch

# ── Data collator ──────────────────────────────────────────────────────────
@dataclass
class DataCollatorCTCWithPadding:
    processor: Any
    padding: Union[bool, str] = True

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_values": feature["input_values"]} for feature in features]
        label_features = [{"input_ids": feature["labels"]}          for feature in features]

        batch = self.processor.feature_extractor.pad(
            input_features,
            padding=self.padding,
            return_tensors="pt",
        )

        labels_batch = self.processor.tokenizer.pad(
            label_features,
            padding=self.padding,
            return_tensors="pt",
        )

        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )
        batch["labels"] = labels

        return batch

data_collator = DataCollatorCTCWithPadding(processor=processor, padding=True)

# ── Quick test on one batch ────────────────────────────────────────────────
test_batch = prepare_dataset({
    "audio": [dataset["train"][0]["audio"]],
    "text":  [dataset["train"][0]["text"]],
})
print("input_values length:", len(test_batch["input_values"][0]))
print("input_length:", test_batch["input_length"])
print("labels:", test_batch["labels"][0][:20])
print("decoded labels:", processor.tokenizer.decode(
    [l for l in test_batch["labels"][0] if l != -100]
))
print("\nData collator:", data_collator)

input_values length: 111776
input_length: [111776]
labels: [29, 6, 2, 33, 6, 29, 33, 2, 32, 14, 6, 28, 2, 6, 29, 18, 35, 14, 2, 11]
decoded labels: لا والو هداك السيد جارنا بروفوكاني بغا يحكرني ولكن نهار لاول يموت لمش دخلتو سوق راسو

Data collator: DataCollatorCTCWithPadding(processor=Wav2Vec2Processor:
- feature_extractor: Wav2Vec2FeatureExtractor {
  "do_normalize": true,
  "feature_extractor_type": "Wav2Vec2FeatureExtractor",
  "feature_size": 1,
  "padding_side": "right",
  "padding_value": 0.0,
  "return_attention_mask": true,
  "sampling_rate": 16000
}

- tokenizer: Wav2Vec2CTCTokenizer(name_or_path='/content/processor', vocab_size=36, model_max_length=1000000000000000019884624838656, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<s>', 'eos_token': '</s>', 'unk_token': '[UNK]', 'pad_token': '[PAD]', 'word_delimiter_token': '|'}, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=True, lstrip=True, single_word=False, normalized=False, special=Fals

In [6]:
# ── Apply preprocessing to full dataset ────────────────────────────────────
cols_to_remove = dataset["train"].column_names
print("Columns to remove:", cols_to_remove)

dataset = dataset.map(
    prepare_dataset,
    remove_columns=cols_to_remove,
    batch_size=128,
    batched=True,
    num_proc=4,
)

print(dataset)

Columns to remove: ['id', 'audio', 'text']


Map (num_proc=4):   0%|          | 0/33338 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/6163 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/5752 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_values', 'input_length', 'labels'],
        num_rows: 33338
    })
    validation: Dataset({
        features: ['input_values', 'input_length', 'labels'],
        num_rows: 6163
    })
    test: Dataset({
        features: ['input_values', 'input_length', 'labels'],
        num_rows: 5752
    })
})
